# Cultural binding — Phase 2 post-hoc analyses

Read-only post-processing of the 8 `_stage4_ko.pkl` files (4 base + 4 instruct).
**No GPU, no model loading, no forward passes** — pure pandas/scipy/statsmodels.

Reproduces the cross-model post-hoc analyses of the paper (behavioural
knowledge-to-action gap, ΔS–ΔK colocalisation, directional differentiation
under knockout; §4.6–§4.7, Appendices K–L), after re-checking every pickle
against the published Tables 1/2/6/7 via the reproduction gate.
Rebuilt from `analysis_phase2.ipynb` (canonical, `cultural-binding-heads-main/`),
cells 0–18. The 8 pickles are searched recursively under
`./results/`, where the pipeline notebooks in this repo write them.

## Layout
1. **Setup** — paths + imports + optional `statsmodels` install
2. **Load & pre-flight** — glob the 8 pickles, assert lengths / unique items / dict format
3. **Reproduction gate** — recompute |ΔS|, |ΔK|, K/S from each pickle and compare to
   published Tables 1/2/6/7. Any model that deviates beyond rounding is FLAGGED and
   excluded from downstream interpretation.
4. **Analysis 1** — knowledge-probe behavioural accuracy (argmax over {a, b, c})
5. **Analysis 2** — per-pair ΔS ~ ΔK correlations: B1 baseline mixed-effects + B2
   knockout colocalisation effect_S vs effect_K (with U→item control)
6. **Analysis 3** — directional differentiation under knockout: P(R | a or b) for
   baseline / R-KO / U-KO, paired t-tests on 66 item-means
7. **Outputs** — `results/phase2/{*.csv, *.png, phase2_summary.pkl}`

## Statistical doctrine (printed alongside every test)
- **Unit of inference = ITEM (n=66)**, never per-pair (n=847): the 847 pairs share
  66 items → pair-level inference is pseudoreplication.
- **PAIRED tests** when conditions are measured on the SAME pairs (baseline vs KO).
- **U→item control reported next to every R→item effect** — specificity is the claim.
- **Robust statistic next to parametric** (Spearman next to Pearson).

## 1. Setup

In [ ]:
# ============================================================
# SETUP: imports, paths
# ============================================================
import os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
import sys
sys.path.insert(0, REPO_ROOT)
import sys
import glob
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy import stats


# statsmodels: needed for mixedlm. Install if missing (Colab usually has it).
try:
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    print(f"statsmodels {sm.__version__} OK")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    print(f"statsmodels {sm.__version__} installed")

# Path correction for the submission repo (validated; was: Colab Drive mount
# with a "./results_regen/" fallback). The 8 stage4_ko pickles are searched
# recursively under ./results/, where the pipeline notebooks write them.
DRIVE_ROOT = os.path.join(REPO_ROOT, "results") + os.sep
print(f"DRIVE_ROOT = {DRIVE_ROOT}")

OUTPUT_DIR = Path(REPO_ROOT) / "results" / "phase2"
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")

## 2. Load 8 pickles + pre-flight asserts

In [ ]:
# ============================================================
# LOAD 8 PICKLES + PRE-FLIGHT ASSERTS
# Key asymmetry: base uses 'gemma', instruct uses 'gemma2'.
# ============================================================
EXPECTED = [
    ("mistral", "base"),     ("llama",  "base"),     ("gemma",  "base"),     ("nemo", "base"),
    ("mistral", "instruct"), ("llama",  "instruct"), ("gemma2", "instruct"), ("nemo", "instruct"),
]

def _find_pickle(drive_root, key, variant):
    fname = f"results_{key}_{variant}_stage4_ko.pkl"
    matches = glob.glob(os.path.join(drive_root, "**", fname), recursive=True)
    return matches[0] if matches else None

stores = {}
found_paths = {}
for key, variant in EXPECTED:
    p = _find_pickle(DRIVE_ROOT, key, variant)
    if p is None:
        print(f"  MISSING: results_{key}_{variant}_stage4_ko.pkl")
        continue
    with open(p, "rb") as f:
        stores[(key, variant)] = pickle.load(f)
    found_paths[(key, variant)] = p
    print(f"  Loaded: {os.path.relpath(p, DRIVE_ROOT)}  ({os.path.getsize(p)//1024} KB)")

if len(stores) < 8:
    raise RuntimeError(f"Found only {len(stores)}/8 pickles. Aborting.")
print(f"\nAll 8 pickles loaded.")


# Pre-flight asserts (stop early on data-contract violations)
print("\nPre-flight checks:")
for (key, variant), store in stores.items():
    label = f"{key:8s} {variant}"
    fac = store["factorial"]
    assert len(fac["items"])     == 847, f"{label}: factorial.items not len 847"
    assert len(fac["assoc_pos"]) == 847, f"{label}: factorial.assoc_pos not len 847"
    assert len(set(fac["items"]))== 66,  f"{label}: unique items != 66 (got {len(set(fac['items']))})"
    assert set(fac["assoc_pos"]).issubset({"a", "b"}), f"{label}: assoc_pos not subset (a,b)"

    bind = store["binding"]
    for lp_key in ("logprobs_match", "logprobs_mismatch",
                   "logprobs_match_Rko", "logprobs_mismatch_Rko",
                   "logprobs_match_Uko", "logprobs_mismatch_Uko"):
        arr = bind.get(lp_key)
        assert arr is not None and len(arr) == 847, f"{label}: binding.{lp_key} missing or wrong length"
        for x in arr[:3]:
            assert isinstance(x, dict) and set(x.keys()) == {"a", "b", "c"}, \
                f"{label}: binding.{lp_key}[0..2] not dict with keys (a,b,c)"

    knw = store["knowledge"]
    for lp_key in ("logprobs_match", "logprobs_mismatch"):
        arr = knw.get(lp_key)
        assert arr is not None and len(arr) == 847, f"{label}: knowledge.{lp_key} missing or wrong length"

    ko = store["knockout"]
    for task in ("binding", "knowledge"):
        for d_key in ("diffs_base", "diffs_B_ko", "diffs_A_ko"):
            arr = np.asarray(ko[task].get(d_key, []))
            assert arr.size == 847, f"{label}: knockout.{task}.{d_key} length {arr.size}, expected 847"
            assert not np.any(np.isnan(arr)), f"{label}: knockout.{task}.{d_key} contains NaN"

    print(f"  {label}: OK")

## 3. Helpers (lifted from pipeline notebooks)

In [ ]:
# ============================================================
# HELPERS lifted from the pipeline notebooks (cleaned)
# ============================================================
def cond_p_R(logprobs_list, assoc_pos):
    """P(R | {a, b}) per pair: softmax over (lp[assoc_pos], lp[other_ab]).
    Conditional probability of picking the associated letter, ignoring c."""
    out = []
    for lps, ap in zip(logprobs_list, assoc_pos):
        lp_R = lps[ap]
        lp_U = lps["b" if ap == "a" else "a"]
        m = max(lp_R, lp_U)
        out.append(np.exp(lp_R - m) / (np.exp(lp_R - m) + np.exp(lp_U - m)))
    return np.array(out)


def to_item_means(per_pair, items):
    """Aggregate per-pair quantity (len 847) to per-item means (len 66)."""
    items = np.asarray(items)
    per_pair = np.asarray(per_pair)
    return np.array([per_pair[items == it].mean() for it in np.unique(items)])


def argmax_pred(logprobs_list):
    """Return per-pair argmax letter ('a', 'b' or 'c') from a list of {a,b,c} dicts."""
    out = []
    for d in logprobs_list:
        # max() over dict picks the key with the largest value.
        out.append(max(d, key=d.__getitem__))
    return np.array(out)


# Quick sanity test of helpers
_test_lps = [{"a": -1.0, "b": -2.0, "c": -3.0}, {"a": -3.0, "b": -1.0, "c": -2.0}]
_test_assoc = np.array(["a", "b"])
assert np.allclose(cond_p_R(_test_lps, _test_assoc),
                   [1/(1+np.exp(-1)), 1/(1+np.exp(-2))]), "cond_p_R math drift"
_test_pred = argmax_pred(_test_lps)
assert _test_pred.tolist() == ["a", "b"]
print("Helpers OK.")

## 4. Reproduction gate

In [ ]:
# ============================================================
# REPRODUCTION GATE — pass before interpreting anything
# Justification: recompute |dS|, |dK|, K/S after KO from each pickle and compare
# to published Tables 1/2/6/7 (SEED=42, fixed factorial). FLAG any model whose
# recomputed value deviates beyond rounding; that model is excluded from analyses.
# ============================================================
# TARGETS: same values as the inline table of the source notebook, now
# imported from common.published_targets (verified identical at curation).
from common.published_targets import TARGETS_BASE, TARGETS_INSTRUCT

TARGETS = {}
for _key, _t in TARGETS_BASE.items():
    TARGETS[(_key, "base")] = {"abs_dS": _t["abs_dS"], "abs_dK": _t["abs_dK"], "KS": _t["KS_ratio"]}
for _key, _t in TARGETS_INSTRUCT.items():
    TARGETS[(_key, "instruct")] = {"abs_dS": _t["abs_dS"], "abs_dK": _t["abs_dK"], "KS": _t["KS_ratio"]}

# Tolerances: rounding precision of published tables.
# base is reported to 3 decimals (max abs ~1) so 5e-3 is loose-rounding;
# instruct is reported to 2 decimals (values up to ~12) so 5e-2 mirrors that.
TOL_ABS = {"base": 5e-3, "instruct": 5e-2}
TOL_KS  = 0.05

print("Justification: recompute |dS|, |dK|, K/S after KO and compare to "
      "published Tables 1/2/6/7. FLAG models exceeding rounding tolerance.")
print()

repro_rows = []
INVALID = set()
for (key, variant), store in stores.items():
    bind_ko = store["knockout"]["binding"]
    knw_ko  = store["knockout"]["knowledge"]
    abs_dS  = abs(float(np.mean(bind_ko["diffs_base"])))
    abs_dK  = abs(float(np.mean(knw_ko["diffs_base"])))
    red_B_S = float(bind_ko["reduction_B_pct"])
    red_B_K = float(knw_ko["reduction_B_pct"])
    KS_KO   = red_B_K / red_B_S if abs(red_B_S) > 1e-6 else float("nan")

    tgt = TARGETS[(key, variant)]
    tol = TOL_ABS[variant]
    devS  = abs(abs_dS - tgt["abs_dS"])
    devK  = abs(abs_dK - tgt["abs_dK"])
    devKS = abs(KS_KO  - tgt["KS"])

    flagged = (devS > tol) or (devK > tol) or (devKS > TOL_KS)
    if flagged:
        INVALID.add((key, variant))

    repro_rows.append({
        "model": key, "variant": variant,
        "abs_dS": round(abs_dS, 4),  "tgt_dS": tgt["abs_dS"], "dev_dS": round(devS, 4),
        "abs_dK": round(abs_dK, 4),  "tgt_dK": tgt["abs_dK"], "dev_dK": round(devK, 4),
        "KS_KO":  round(KS_KO, 3),   "tgt_KS": tgt["KS"],     "dev_KS": round(devKS, 3),
        "STATUS": "INVALID" if flagged else "OK",
    })

repro_df = pd.DataFrame(repro_rows)
repro_df = repro_df.set_index(["variant", "model"]).sort_index()
print(repro_df.to_string())
print()
if INVALID:
    print(f"⚠️ INVALID model(s) — excluded from downstream interpretation: {sorted(INVALID)}")
else:
    print("✓ All 8 model×variant rows PASS the reproduction gate.")

## 4b. Table 2 significance markers (added at curation)

Recomputes the significance markers (†) of Table 2 at the unit stated in the
paper — paired t-tests on item-level means (n = 66): **one-sided** for the
R→item knockout, whose pre-specified direction is a decrease in |ΔS|
(§3.4: "|ΔS| should decrease"), **two-sided** for the U→item control, for
which no decrease is predicted. Consistent with the one-sided empirical
p-value already used for the random-head baseline (§4.3).


In [ ]:
# ============================================================
# 4b. TABLE 2 SIGNIFICANCE MARKERS (added at curation)
# ------------------------------------------------------------
# Paired t-tests on item-level means (n = 66), from the per-pair
# ΔS diffs stored in the stage4 pickles:
#   R→item — one-sided (pre-specified direction: the knockout
#            weakens binding, i.e. ΔS_KO is less negative than
#            ΔS_baseline);
#   U→item — two-sided (control; no decrease predicted).
# reduction_*_pct are the values stored by the pipeline notebooks
# (positive = |ΔS| reduced).
# ============================================================
print("=" * 74)
print("TABLE 2 SIGNIFICANCE — paired t-tests on item-level means (n = 66)")
print("R→item: one-sided (pre-specified direction) | U→item: two-sided")
print("=" * 74)

table2_sig = {}
for (key, variant), store in sorted(stores.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    items = np.array(store["factorial"]["items"])
    ko = store["knockout"]["binding"]
    m_base = to_item_means(ko["diffs_base"], items)
    row = {}
    for cond, dkey, rkey in (("R", "diffs_B_ko", "reduction_B_pct"),
                             ("U", "diffs_A_ko", "reduction_A_pct")):
        m_ko = to_item_means(ko[dkey], items)
        t, p_two = stats.ttest_rel(m_base, m_ko)
        if cond == "R":
            p = p_two / 2.0 if (m_ko - m_base).mean() > 0 else 1.0 - p_two / 2.0
        else:
            p = p_two
        row[cond] = {"t": float(t), "p": float(p), "reduction_pct": float(ko[rkey])}
    table2_sig[(key, variant)] = row
    _fmt = lambda c: f"p={row[c]['p']:.2e}" + ("" if row[c]["p"] < 0.05 else "  † n.s.")
    print(f"{key:8} {variant:9}  R→item red {row['R']['reduction_pct']:+6.1f}%  {_fmt('R'):22}"
          f"  U→item red {row['U']['reduction_pct']:+6.1f}%  {_fmt('U')}")


## 5. Analysis 1 — knowledge-probe behavioural accuracy

In [ ]:
# ============================================================
# ANALYSIS 1 — Knowledge-probe behavioural accuracy
# Justification: pred = argmax over {a, b, c} per pair; aggregate the indicator
# (pred == assoc_pos | pred != c) to item-level (n=66) and ttest_1samp vs 0.5,
# i.e. chance when the model differentiates at all (excluding "c" abstentions).
# ============================================================
print("Justification: per-pair pred=argmax(a,b,c). MATCH metrics: %pick_R, %pick_c, "
      "%pick_U. Conditional accuracy = mean(pred==assoc_pos | pred!=c). Aggregate to "
      "n=66 item-means, ttest_1samp vs 0.5 (chance R-vs-U conditional on differentiating).")
print()

a1_rows = []
for (key, variant), store in stores.items():
    if (key, variant) in INVALID:
        continue
    items     = np.array(store["factorial"]["items"])
    assoc_pos = np.array(store["factorial"]["assoc_pos"])
    unique    = np.unique(items)

    for task in ("binding", "knowledge"):
        lp_match = store[task]["logprobs_match"]
        lp_mism  = store[task]["logprobs_mismatch"]
        pred_match = argmax_pred(lp_match)
        pred_mism  = argmax_pred(lp_mism)

        # MATCH per-pair indicators
        pick_R = (pred_match == assoc_pos).astype(float)
        pick_c = (pred_match == "c").astype(float)
        pick_U = ((pred_match != assoc_pos) & (pred_match != "c")).astype(float)

        # MATCH item-means
        pick_R_item = to_item_means(pick_R, items)
        pick_c_item = to_item_means(pick_c, items)
        pick_U_item = to_item_means(pick_U, items)

        # Conditional differentiation accuracy: mean(pred==assoc | pred!=c) per item
        diff_mask = (pred_match != "c")
        cond_acc_item = np.array([
            ( (pred_match[(items == it) & diff_mask] == assoc_pos[(items == it) & diff_mask]).mean()
              if ((items == it) & diff_mask).sum() > 0 else np.nan )
            for it in unique
        ])
        valid = ~np.isnan(cond_acc_item)
        if valid.sum() >= 5:
            t_cond, p_cond = stats.ttest_1samp(cond_acc_item[valid], 0.5)
        else:
            t_cond, p_cond = float("nan"), float("nan")

        # MISMATCH: %pick_c (knowledge should be high; binding less so)
        pick_c_mism_item = to_item_means((pred_mism == "c").astype(float), items)

        a1_rows.append({
            "model": key, "variant": variant, "task": task,
            "match_pick_R":  round(float(pick_R_item.mean()), 4),
            "match_pick_c":  round(float(pick_c_item.mean()), 4),
            "match_pick_U":  round(float(pick_U_item.mean()), 4),
            "mism_pick_c":   round(float(pick_c_mism_item.mean()), 4),
            "cond_acc_item": round(float(np.nanmean(cond_acc_item)), 4),
            "cond_acc_t":    round(float(t_cond), 3),
            "cond_acc_p":    float(p_cond),
        })

a1_df = pd.DataFrame(a1_rows)
print("Analysis 1 — knowledge & binding argmax accuracy (item-means, conditional t vs 0.5):")
print(a1_df.to_string(index=False))


# HEADLINE: knowledge − binding %pick_R = behavioural knowledge-to-action gap
print()
gap_rows = []
for (key, variant) in stores.keys():
    if (key, variant) in INVALID:
        continue
    knw_R  = float(a1_df[(a1_df.model == key) & (a1_df.variant == variant) & (a1_df.task == "knowledge")]["match_pick_R"].iloc[0])
    bind_R = float(a1_df[(a1_df.model == key) & (a1_df.variant == variant) & (a1_df.task == "binding")]["match_pick_R"].iloc[0])
    gap_rows.append({
        "model": key, "variant": variant,
        "knowledge_pick_R": round(knw_R, 4),
        "binding_pick_R":   round(bind_R, 4),
        "gap_K_minus_B":    round(knw_R - bind_R, 4),
    })
gap_df = pd.DataFrame(gap_rows).sort_values(["variant", "model"])
print("HEADLINE — knowledge-to-action gap (knowledge − binding %pick_R, item-means):")
print(gap_df.to_string(index=False))
print()
print("Note: in base models the gap is typically small (both tasks near chance); the meaningful gap is the instruct row.")

## 6. Analysis 2 — per-pair ΔS ~ ΔK correlations

In [ ]:
# ============================================================
# ANALYSIS 2 — Per-item dS ~ dK correlations (CORRECTED B2)
#
# B1 (UNCHANGED) baseline association: mixedlm("absS ~ absK", groups=item)
#   Justification: 847 pairs nested in 66 items; mixedlm absorbs item-level
#   dependence. Pearson/Spearman at pair AND item level reported for transparency.
#
# B2 (REPLACED) knockout colocalisation — sign-consistent STRENGTH effect.
#   Aggregate to ITEM LEVEL first (n=66). Define:
#       effR_str = abs(base_it) - abs(Rko_it)     # positive = R-KO reduced strength
#       effU_str = abs(base_it) - abs(Uko_it)     # positive = U-KO reduced strength
#       effR_sig = base_it - Rko_it ; effU_sig = base_it - Uko_it     (reference)
#   Three separate questions, never collapsed into a single "spec?" verdict:
#     (1) SIGN DISSOCIATION: mean(effR_str) > 0 expected, mean(effU_str) < 0 expected.
#         ttest_1samp on the 66 item-level effects vs 0.
#     (2) SHARED-CIRCUIT under R: corr(effR_str_S, effR_str_K), item level.
#     (3) U-colocalisation: corr(effU_str_S, effU_str_K). DESCRIPTIVE, not pass/fail.
# ============================================================
def _marg_R2(mdf, x):
    """Quick Nakagawa marginal R^2 for a single fixed effect."""
    try:
        slope = float(mdf.params.get("absK", np.nan))
        var_fixed  = (slope ** 2) * np.var(x, ddof=0)
        var_random = float(mdf.cov_re.iloc[0, 0]) if mdf.cov_re.size else 0.0
        var_resid  = float(mdf.scale)
        return var_fixed / (var_fixed + var_random + var_resid)
    except Exception:
        return float("nan")


print("B1 Justification: mixedlm(absS ~ absK, groups=item) handles the 66-item clustering.")
print("B2 Justification: sign-consistent strength effect (abs(base) - abs(KO)) reveals "
      "specificity in the MEAN (R reduces strength, U strengthens), and tests SHARED-CIRCUIT "
      "colocalisation separately. U-colocalisation is descriptive, not a pass/fail control.")
print()

a2_rows = []
for (key, variant), store in stores.items():
    if (key, variant) in INVALID:
        continue
    items = np.array(store["factorial"]["items"])

    # ----- B1 baseline absS ~ absK -----
    absS = np.abs(np.asarray(store["knockout"]["binding"]["diffs_base"]))
    absK = np.abs(np.asarray(store["knockout"]["knowledge"]["diffs_base"]))

    r_p_pair, p_p_pair = stats.pearsonr(absS, absK)
    rho_pair, p_rho_pair = stats.spearmanr(absS, absK)

    absS_it = to_item_means(absS, items)
    absK_it = to_item_means(absK, items)
    r_p_item, p_p_item = stats.pearsonr(absS_it, absK_it)
    rho_item, p_rho_item = stats.spearmanr(absS_it, absK_it)

    df_b1 = pd.DataFrame({"absS": absS, "absK": absK, "item": items})
    mixed_slope = float("nan"); mixed_ci_lo = mixed_ci_hi = float("nan")
    mixed_R2 = float("nan"); mixed_status = "FAIL"
    try:
        mdf = smf.mixedlm("absS ~ absK", df_b1, groups=df_b1["item"]).fit(reml=True, method=["lbfgs"])
        mixed_slope = float(mdf.params["absK"])
        ci = mdf.conf_int().loc["absK"]
        mixed_ci_lo = float(ci[0]); mixed_ci_hi = float(ci[1])
        mixed_R2 = _marg_R2(mdf, absK)
        mixed_status = "converged" if mdf.converged else "non-converged"
    except Exception as e:
        lr = stats.linregress(absK_it, absS_it)
        mixed_slope = float(lr.slope)
        mixed_ci_lo = float(lr.slope - 1.96 * lr.stderr)
        mixed_ci_hi = float(lr.slope + 1.96 * lr.stderr)
        mixed_R2 = float(lr.rvalue ** 2)
        mixed_status = f"OLS_items_fallback ({e!r})"

    # ----- B2 CORRECTED: item-level strength and signed effects -----
    base_S_it = to_item_means(np.asarray(store["knockout"]["binding"]["diffs_base"]), items)
    Rko_S_it  = to_item_means(np.asarray(store["knockout"]["binding"]["diffs_B_ko"]), items)
    Uko_S_it  = to_item_means(np.asarray(store["knockout"]["binding"]["diffs_A_ko"]), items)
    base_K_it = to_item_means(np.asarray(store["knockout"]["knowledge"]["diffs_base"]), items)
    Rko_K_it  = to_item_means(np.asarray(store["knockout"]["knowledge"]["diffs_B_ko"]), items)
    Uko_K_it  = to_item_means(np.asarray(store["knockout"]["knowledge"]["diffs_A_ko"]), items)

    effR_str_S = np.abs(base_S_it) - np.abs(Rko_S_it)   # >0 → R-KO weakened binding strength
    effU_str_S = np.abs(base_S_it) - np.abs(Uko_S_it)   # expected <0 → U-KO strengthened binding
    effR_str_K = np.abs(base_K_it) - np.abs(Rko_K_it)
    effU_str_K = np.abs(base_K_it) - np.abs(Uko_K_it)

    effR_sig_S = base_S_it - Rko_S_it                   # signed (reference)
    effU_sig_S = base_S_it - Uko_S_it
    effR_sig_K = base_K_it - Rko_K_it
    effU_sig_K = base_K_it - Uko_K_it

    # (1) SIGN DISSOCIATION — ttest_1samp vs 0 on the 66 item-level effects
    t_R_str_S, p_R_str_S = stats.ttest_1samp(effR_str_S, 0.0)
    t_U_str_S, p_U_str_S = stats.ttest_1samp(effU_str_S, 0.0)
    t_R_str_K, p_R_str_K = stats.ttest_1samp(effR_str_K, 0.0)
    t_U_str_K, p_U_str_K = stats.ttest_1samp(effU_str_K, 0.0)

    # (2) shared-circuit under R: corr(effR_str_S, effR_str_K)
    R_r_str, R_p_r_str   = stats.pearsonr(effR_str_S, effR_str_K)
    R_rho_str, R_p_rho_str = stats.spearmanr(effR_str_S, effR_str_K)

    # (3) colocalisation under U (descriptive)
    U_r_str, U_p_r_str   = stats.pearsonr(effU_str_S, effU_str_K)
    U_rho_str, U_p_rho_str = stats.spearmanr(effU_str_S, effU_str_K)

    # Signed correlations (reference only)
    R_r_sig, _ = stats.pearsonr(effR_sig_S, effR_sig_K)
    U_r_sig, _ = stats.pearsonr(effU_sig_S, effU_sig_K)

    a2_rows.append({
        "model": key, "variant": variant,
        # B1
        "B1_pearson_pair":  round(r_p_pair, 3),  "B1_p_pearson_pair":  float(p_p_pair),
        "B1_spear_pair":    round(rho_pair, 3),  "B1_p_spear_pair":    float(p_rho_pair),
        "B1_pearson_item":  round(r_p_item, 3),  "B1_p_pearson_item":  float(p_p_item),
        "B1_spear_item":    round(rho_item, 3),  "B1_p_spear_item":    float(p_rho_item),
        "B1_mix_slope":     round(mixed_slope, 4),
        "B1_mix_CI":        (round(mixed_ci_lo, 4), round(mixed_ci_hi, 4)),
        "B1_mix_marg_R2":   round(mixed_R2, 4),
        "B1_mix_status":    mixed_status,
        # B2 (1) sign-dissociation in MEAN strength
        "mean_effR_str_S":  round(float(effR_str_S.mean()), 5),
        "t_R_str_S":        round(float(t_R_str_S), 2), "p_R_str_S": float(p_R_str_S),
        "mean_effU_str_S":  round(float(effU_str_S.mean()), 5),
        "t_U_str_S":        round(float(t_U_str_S), 2), "p_U_str_S": float(p_U_str_S),
        "mean_effR_str_K":  round(float(effR_str_K.mean()), 5),
        "t_R_str_K":        round(float(t_R_str_K), 2), "p_R_str_K": float(p_R_str_K),
        "mean_effU_str_K":  round(float(effU_str_K.mean()), 5),
        "t_U_str_K":        round(float(t_U_str_K), 2), "p_U_str_K": float(p_U_str_K),
        # B2 (2) shared-circuit colocalisation under R
        "R_coloc_r_str":    round(float(R_r_str), 3),  "R_coloc_p_r_str":   float(R_p_r_str),
        "R_coloc_rho_str":  round(float(R_rho_str), 3),"R_coloc_p_rho_str": float(R_p_rho_str),
        # B2 (3) U-colocalisation (descriptive)
        "U_coloc_r_str":    round(float(U_r_str), 3),  "U_coloc_p_r_str":   float(U_p_r_str),
        "U_coloc_rho_str":  round(float(U_rho_str), 3),"U_coloc_p_rho_str": float(U_p_rho_str),
        # Signed reference
        "R_coloc_r_signed": round(float(R_r_sig), 3),
        "U_coloc_r_signed": round(float(U_r_sig), 3),
    })

a2_df = pd.DataFrame(a2_rows)

print("B1 baseline association (absS ~ absK):")
print(a2_df[["model", "variant", "B1_mix_slope", "B1_mix_CI", "B1_mix_marg_R2",
             "B1_mix_status", "B1_pearson_item", "B1_spear_item"]].to_string(index=False))
print()
print("B2 (1) SIGN DISSOCIATION — mean strength change at item level (t vs 0, n=66):")
print("       Expectation: mean(effR_str) > 0 (R weakens); mean(effU_str) < 0 (U strengthens).")
b2_1 = a2_df[["model", "variant",
              "mean_effR_str_S", "t_R_str_S", "p_R_str_S",
              "mean_effU_str_S", "t_U_str_S", "p_U_str_S",
              "mean_effR_str_K", "t_R_str_K", "p_R_str_K",
              "mean_effU_str_K", "t_U_str_K", "p_U_str_K"]]
print(b2_1.to_string(index=False))
print()
print("B2 (2) SHARED-CIRCUIT colocalisation under R->item (item-level Pearson & Spearman on STRENGTH effect):")
print(a2_df[["model", "variant", "R_coloc_r_str", "R_coloc_p_r_str",
             "R_coloc_rho_str", "R_coloc_p_rho_str", "R_coloc_r_signed"]].to_string(index=False))
print()
print("B2 (3) U->item colocalisation (DESCRIPTIVE; redistribution effects also covary):")
print(a2_df[["model", "variant", "U_coloc_r_str", "U_coloc_p_r_str",
             "U_coloc_rho_str", "U_coloc_p_rho_str", "U_coloc_r_signed"]].to_string(index=False))

# One-line per-model interpretation
print()
print("Per-model interpretation:")
for r in a2_df.to_dict("records"):
    sd_S = ("R↓ strength" if (r["t_R_str_S"] > 0 and r["p_R_str_S"] < 0.05)
            else ("R↑ strength" if (r["t_R_str_S"] < 0 and r["p_R_str_S"] < 0.05) else "R≈0 strength"))
    su_S = ("U↑ strength" if (r["t_U_str_S"] < 0 and r["p_U_str_S"] < 0.05)
            else ("U↓ strength" if (r["t_U_str_S"] > 0 and r["p_U_str_S"] < 0.05) else "U≈0 strength"))
    sd_K = ("K↓" if (r["t_R_str_K"] > 0 and r["p_R_str_K"] < 0.05) else ("K↑" if (r["t_R_str_K"] < 0 and r["p_R_str_K"] < 0.05) else "K≈0"))
    coloc_R = f"R-coloc r={r['R_coloc_r_str']:+.2f}"
    coloc_U = f"U-coloc r={r['U_coloc_r_str']:+.2f}"
    print(f"  {r['model']:8s} {r['variant']:8s} | sign-dissoc S: {sd_S}, {su_S} | sign-dissoc K under R: {sd_K} | {coloc_R} | {coloc_U}")

## 7. Analysis 3 — directional differentiation under knockout

In [ ]:
# ============================================================
# ANALYSIS 3 — Directional differentiation under knockout
# Justification: paired t-test on n=66 item-means (within-item paired design).
# Predictions:
#   baseline vs R->item KO: significant DROP toward 0.5 (heads carry direction)
#   baseline vs U->item KO: NOT significant / much weaker (specificity to R)
# Note: base baseline pR is ~0.51-0.55 (near chance); expect tiny / null effects.
# Instruct (~0.60-0.74) is where the drop is meaningful — interpret base cautiously.
# ============================================================
print("Justification: paired t-test on n=66 item-means: baseline vs R->KO and "
      "baseline vs U->KO. Same convention as paper's cluster-corrected KO tests.")
print()

a3_rows = []
for (key, variant), store in stores.items():
    if (key, variant) in INVALID:
        continue
    items     = np.array(store["factorial"]["items"])
    assoc_pos = np.array(store["factorial"]["assoc_pos"])

    pR_base = cond_p_R(store["binding"]["logprobs_match"],     assoc_pos)
    pR_Rko  = cond_p_R(store["binding"]["logprobs_match_Rko"], assoc_pos)
    pR_Uko  = cond_p_R(store["binding"]["logprobs_match_Uko"], assoc_pos)

    pR_base_it = to_item_means(pR_base, items)
    pR_Rko_it  = to_item_means(pR_Rko,  items)
    pR_Uko_it  = to_item_means(pR_Uko,  items)

    t_R, p_R = stats.ttest_rel(pR_base_it, pR_Rko_it)
    t_U, p_U = stats.ttest_rel(pR_base_it, pR_Uko_it)

    a3_rows.append({
        "model": key, "variant": variant,
        "pR_baseline":           round(float(pR_base_it.mean()), 4),
        "pR_Rko":                round(float(pR_Rko_it.mean()), 4),
        "pR_Uko":                round(float(pR_Uko_it.mean()), 4),
        "delta_base_minus_Rko":  round(float(pR_base_it.mean() - pR_Rko_it.mean()), 4),
        "delta_base_minus_Uko":  round(float(pR_base_it.mean() - pR_Uko_it.mean()), 4),
        "paired_t_R":            round(float(t_R), 3),
        "paired_p_R":            float(p_R),
        "paired_t_U":            round(float(t_U), 3),
        "paired_p_U":            float(p_U),
    })

a3_df = pd.DataFrame(a3_rows).sort_values(["variant", "model"])
print("Analysis 3 — Directional differentiation under KO (item-mean P(R | a or b)):")
print(a3_df.to_string(index=False))

## 8. Figures

In [ ]:
# ============================================================
# FIGURES
# ============================================================
plt.rcParams["figure.dpi"] = 100

# --- Fig 1: behavioural gap bar chart (knowledge vs binding %pick_R) ---
g = gap_df.copy().reset_index(drop=True)
xs = np.arange(len(g))
w = 0.38
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(xs - w/2, g["binding_pick_R"],    w, label="binding %pick_R")
ax.bar(xs + w/2, g["knowledge_pick_R"],  w, label="knowledge %pick_R")
ax.set_xticks(xs); ax.set_xticklabels([f"{m}/{v}" for m, v in zip(g["model"], g["variant"])], rotation=45, ha="right")
ax.set_ylabel("%pick_R (item-mean)")
ax.set_title("Knowledge-to-action gap: argmax-over-(a,b,c) %pick_R per model/variant")
ax.axhline(1/3, color="gray", linestyle="--", alpha=0.6, label="chance, 3 options (1/3)")
ax.legend()
plt.tight_layout()
fig_path_1 = OUTPUT_DIR / "fig_behavioural_gap.png"
fig.savefig(fig_path_1, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path_1}")

# --- Fig 2: strength-effect colocalisation, R-KO and U-KO clearly distinguished ---
valid_keys = [k for k in stores.keys() if k not in INVALID]
n = len(valid_keys)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.6*rows), squeeze=False)
for ax_i, (key, variant) in enumerate(valid_keys):
    ax = axes[ax_i // cols, ax_i % cols]
    store = stores[(key, variant)]
    items = np.array(store["factorial"]["items"])
    base_S = to_item_means(np.asarray(store["knockout"]["binding"]["diffs_base"]), items)
    Rko_S  = to_item_means(np.asarray(store["knockout"]["binding"]["diffs_B_ko"]), items)
    Uko_S  = to_item_means(np.asarray(store["knockout"]["binding"]["diffs_A_ko"]), items)
    base_K = to_item_means(np.asarray(store["knockout"]["knowledge"]["diffs_base"]), items)
    Rko_K  = to_item_means(np.asarray(store["knockout"]["knowledge"]["diffs_B_ko"]), items)
    Uko_K  = to_item_means(np.asarray(store["knockout"]["knowledge"]["diffs_A_ko"]), items)

    effR_str_S = np.abs(base_S) - np.abs(Rko_S)
    effU_str_S = np.abs(base_S) - np.abs(Uko_S)
    effR_str_K = np.abs(base_K) - np.abs(Rko_K)
    effU_str_K = np.abs(base_K) - np.abs(Uko_K)

    ax.scatter(effU_str_S, effU_str_K, s=22, alpha=0.5, c="gray",
               edgecolors="none", label="U->item (redistribution)")
    ax.scatter(effR_str_S, effR_str_K, s=22, alpha=0.7, c="C3",
               edgecolors="none", label="R->item (specific knockout)")
    ax.axhline(0, color="k", lw=0.4); ax.axvline(0, color="k", lw=0.4)
    rR, _ = stats.pearsonr(effR_str_S, effR_str_K)
    rU, _ = stats.pearsonr(effU_str_S, effU_str_K)
    ax.set_title(f"{key}/{variant}\nR r={rR:+.2f} | U r={rU:+.2f}", fontsize=9)
    if ax_i == 0:
        ax.legend(fontsize=8, loc="best")
    ax.set_xlabel("strength effect on S")
    ax.set_ylabel("strength effect on K")
for j in range(n, rows*cols):
    axes[j // cols, j % cols].axis("off")
plt.suptitle("B2 colocalisation: strength effect (|base| - |KO|), item-level, R vs U overlaid", y=1.02)
plt.tight_layout()
fig_path_2 = OUTPUT_DIR / "fig_colocalisation.png"
fig.savefig(fig_path_2, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path_2}")

# --- Fig 3: directional pR baseline / R-KO / U-KO grouped bars ---
a3 = a3_df.copy().reset_index(drop=True)
xs = np.arange(len(a3))
w = 0.27
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(xs - w, a3["pR_baseline"], w, label="baseline")
ax.bar(xs,     a3["pR_Rko"],      w, label="R->item KO", color="C3")
ax.bar(xs + w, a3["pR_Uko"],      w, label="U->item KO (control)", color="gray")
ax.axhline(0.5, color="k", lw=0.5, ls="--", label="chance (0.5)")
ax.set_xticks(xs); ax.set_xticklabels([f"{m}/{v}" for m, v in zip(a3["model"], a3["variant"])], rotation=45, ha="right")
ax.set_ylabel("P(R | a or b)  (item-mean)")
ax.set_title("Directional differentiation under knockout")
ax.legend(fontsize=8)
plt.tight_layout()
fig_path_3 = OUTPUT_DIR / "fig_directional_under_KO.png"
fig.savefig(fig_path_3, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path_3}")

## 9. Save outputs + surprise narrator

In [ ]:
# ============================================================
# SAVE OUTPUTS
# ============================================================
a1_df.to_csv(OUTPUT_DIR / "knowledge_argmax_accuracy.csv",   index=False)
gap_df.to_csv(OUTPUT_DIR / "knowledge_argmax_gap.csv",       index=False)
a2_df.to_csv(OUTPUT_DIR / "deltaS_deltaK_correlations.csv", index=False)
a3_df.to_csv(OUTPUT_DIR / "directional_under_knockout.csv",  index=False)

summary = {
    "drive_paths":       {f"{k[0]}_{k[1]}": p for k, p in found_paths.items()},
    "reproduction_gate": repro_df.reset_index().to_dict(orient="records"),
    "invalid_models":    sorted([(k, v) for (k, v) in INVALID]),
    "analysis1_argmax":  a1_df.to_dict(orient="records"),
    "analysis1_gap":     gap_df.to_dict(orient="records"),
    "analysis2_corr":    a2_df.to_dict(orient="records"),
    "analysis3_dir":     a3_df.to_dict(orient="records"),
    "tolerances":        {"TOL_ABS": {"base": 5e-3, "instruct": 5e-2}, "TOL_KS": 0.05},
}
with open(OUTPUT_DIR / "phase2_summary.pkl", "wb") as f:
    pickle.dump(summary, f)
print(f"Wrote CSVs + phase2_summary.pkl to {OUTPUT_DIR}/")


# ============================================================
# FLAG SURPRISES — auto-narrate anything worth a second look
# ============================================================
flags = []

# (i) effect_S~effect_K near zero (broken colocalisation)
for r in a2_df.to_dict(orient="records"):
    if abs(r["R_coloc_r_str"]) < 0.1:
        flags.append(f"{r['model']}/{r['variant']}: R-coloc r(strength,item) ~ {r['R_coloc_r_str']:+.2f}  (near zero — heterogeneous dual role under R-KO)")

# (ii) U-KO drops pR as much as R-KO (specificity broken)
for r in a3_df.to_dict(orient="records"):
    if r["delta_base_minus_Uko"] >= 0.5 * r["delta_base_minus_Rko"] and r["delta_base_minus_Rko"] > 0.01:
        flags.append(f"{r['model']}/{r['variant']}: U-KO drop ({r['delta_base_minus_Uko']:+.3f}) is ≥50% of R-KO drop ({r['delta_base_minus_Rko']:+.3f}) — specificity claim weakens")

# (iii) instruct model with low knowledge %pick_R
for r in a1_df.to_dict(orient="records"):
    if r["variant"] == "instruct" and r["task"] == "knowledge" and r["match_pick_R"] < 0.55:
        flags.append(f"{r['model']}/instruct: knowledge %pick_R = {r['match_pick_R']:.2f} (unexpectedly low for instruct)")

# (iv) base model with implausibly high binding %pick_R
for r in a1_df.to_dict(orient="records"):
    if r["variant"] == "base" and r["task"] == "binding" and r["match_pick_R"] > 0.75:
        flags.append(f"{r['model']}/base: binding %pick_R = {r['match_pick_R']:.2f} (high for base; verify against repro gate)")

if flags:
    print("\nFLAGS (auto-narrated):")
    for f in flags:
        print(f"  - {f}")
else:
    print("\nNo automatic flags raised.")